In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [3]:
import numpy as np
import pandas as pd
import pyreadr
from pathlib import Path
from tqdm import tqdm
from scipy.special import logsumexp

# ================================================================
# CONFIG
# ================================================================
BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# LOAD DATA (NON-ISOLATED)
# ================================================================
snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

y_full = snow.iloc[:, 2:].to_numpy()
coords_full = snow.iloc[:, :2].to_numpy()

y = np.delete(y_full, no_nbs, axis=0)
coords = np.delete(coords_full, no_nbs, axis=0)

S, T = y.shape
print(f"S = {S}, T = {T}")

# ================================================================
# GLOBAL TIME VARIABLES (MATCH MCMC)
# ================================================================
t_raw = np.arange(1, T + 1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std(ddof=0)

# ================================================================
# COVARIATES
# ================================================================
cov_bym = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
    t_trend, t_trend
])   # (T, 8)

cov_iid = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period),
    t_trend
])   # (T, 4)

# ================================================================
# FACTOR COMPONENTS (GLOBAL SCALE, MATCH BYM+factor)
# ================================================================
# latitude
lat_raw = coords[:, 1]
lats = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

# elevation
curr_elev = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
elev = (curr_elev - curr_elev.mean()) / curr_elev.std(ddof=1)

# temperature
snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp = np.delete(temp_full, no_nbs, axis=0)
temp_scaled = (temp - temp.mean()) / temp.std(ddof=0)

# ================================================================
# LOAD POSTERIOR SAMPLES
# ================================================================
iid01  = np.load(BASE_DIR / "ind01_noIso.npz")["all_theta"]
iid10  = np.load(BASE_DIR / "ind10_noIso.npz")["all_theta"]

bym01  = np.load(BASE_DIR / "bym01_noIso_final.npz")["all_theta"]
bym10  = np.load(BASE_DIR / "bym10_noIso_final.npz")["all_theta"]

bymf01 = np.load(BASE_DIR / "bym_factor_01_noiso.npz")["all_theta"]
bymf10 = np.load(BASE_DIR / "bym_factor_10_noiso.npz")["all_theta"]


iid01  = np.ascontiguousarray(iid01)
iid10  = np.ascontiguousarray(iid10)
bym01  = np.ascontiguousarray(bym01)
bym10  = np.ascontiguousarray(bym10)
bymf01 = np.ascontiguousarray(bymf01)
bymf10 = np.ascontiguousarray(bymf10)

M = iid01.shape[1]
print("Posterior draws M =", M)

# ================================================================
# POINTWISE CONDITIONAL LOG-LIKELIHOOD
# ================================================================
def loglik_pointwise(
    y, theta01, theta10,
    cov01, cov10,
    lats=None, elev=None, temp_scaled=None, t_trend=None
):

    S, T = y.shape
    K = cov01.shape[1]
    use_factor = lats is not None

    out = np.empty(S * (T-1))
    idx = 0

    for t in range(1, T):
        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(K):
            eta01 += cov01[t, k] * theta01[k*S:(k+1)*S]
            eta10 += cov10[t, k] * theta10[k*S:(k+1)*S]

        if use_factor:
            g01 = theta01[K*S:K*S+3]
            g10 = theta10[K*S:K*S+3]

            eta01 += t_trend[t] * (
                g01[0]*lats + g01[1]*elev + g01[2]*temp_scaled[:, t]
            )
            eta10 += t_trend[t] * (
                g10[0]*lats + g10[1]*elev + g10[2]*temp_scaled[:, t]
            )

        p01 = 1.0 / (1.0 + np.exp(-eta01))
        p10 = 1.0 / (1.0 + np.exp(-eta10))

        prob = np.where(y[:, t-1] == 0, p01, 1.0 - p10)

        ll = (
            y[:, t] * np.log(prob + 1e-12)
            + (1 - y[:, t]) * np.log(1 - prob + 1e-12)
        )

        out[idx:idx+S] = ll
        idx += S

    return out

# ================================================================
# FAST STREAMING WAIC-2
# ================================================================
def compute_waic_streaming(
    label,
    theta01, theta10,
    cov01, cov10,
    use_factor=False
):
    S, T = y.shape
    n_pt = S * (T-1)

    sum_exp = np.zeros(n_pt)
    sum_ll  = np.zeros(n_pt)
    sum_ll2 = np.zeros(n_pt)

    for m in tqdm(range(M), desc=f"WAIC ({label})"):
        ll = loglik_pointwise(
            y,
            theta01[:, m],
            theta10[:, m],
            cov01, cov10,
            lats if use_factor else None,
            elev if use_factor else None,
            temp_scaled if use_factor else None,
            t_trend if use_factor else None
        )

        sum_exp += np.exp(ll)
        sum_ll  += ll
        sum_ll2 += ll**2

    # lppd
    lppd = np.sum(np.log(sum_exp / M))

    # pWAIC (variance form)
    var = (sum_ll2 - (sum_ll**2) / M) / (M - 1)
    p_waic = np.sum(var)

    waic = -2 * (lppd - p_waic)

    return {
        "WAIC": waic,
        "lppd": lppd,
        "p_WAIC": p_waic
    }

# ================================================================
# RUN ALL THREE
# ================================================================
out_iid  = compute_waic_streaming(
    "IID", iid01, iid10, cov_iid, cov_iid
)

out_bym  = compute_waic_streaming(
    "BYM", bym01, bym10, cov_bym, cov_bym
)

out_bymf = compute_waic_streaming(
    "BYM+factor",
    bymf01, bymf10,
    cov_bym, cov_bym,
    use_factor=True
)

# ================================================================
# PRINT RESULTS
# ================================================================
print("\n===== WAIC COMPARISON =====\n")

for name, out in zip(
    ["IID", "BYM", "BYM + factor"],
    [out_iid, out_bym, out_bymf]
):
    print(name)
    for k, v in out.items():
        print(f"  {k}: {v:.3f}")
    print()

print("ΔWAIC (BYM − IID):",  out_bym["WAIC"]  - out_iid["WAIC"])
print("ΔWAIC (BYM+ − BYM):", out_bymf["WAIC"] - out_bym["WAIC"])
print("ΔWAIC (BYM+ − IID):", out_bymf["WAIC"] - out_iid["WAIC"])


S = 1601, T = 2704
Posterior draws M = 1000


WAIC (BYM+factor): 100%|██████████| 1000/1000 [08:54<00:00,  1.87it/s]


===== WAIC COMPARISON =====

IID
  WAIC: 1307955.618
  lppd: -647008.269
  p_WAIC: 6969.540

BYM
  WAIC: 1301743.979
  lppd: -647957.143
  p_WAIC: 2914.847

BYM + factor
  WAIC: 1293295.889
  lppd: -643740.404
  p_WAIC: 2907.540

ΔWAIC (BYM − IID): -6211.638335492229
ΔWAIC (BYM+ − BYM): -8448.09044012148
ΔWAIC (BYM+ − IID): -14659.728775613708
